In [ ]:
# [Cell 20: The V4 Master Ensemble - Poisson + Baseline Weighted Training]
import lightgbm as lgb
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error

# 1. SETUP DATA
# We ensure the columns match the output from the new Cell 2
X_v4 = windowed_df.drop(columns=['target'])
y_v4 = windowed_df['target'].copy()

# Poisson cannot handle negative labels (red cards/conceding)
y_v4_clipped = np.maximum(y_v4, 0)

# 2. WEIGHTED TRAINING (Star-Priority)
# We give 2.5x importance to high-scoring matches (8+) 
# to help the model learn the "peaks" of elite players.
weights = np.where(y_v4_clipped >= 8, 2.5, 1.0) 

# 3. POISSON REGRESSION PARAMETERS
lgbm_v4_params = {
    'objective': 'poisson',
    'n_estimators': 1200,
    'learning_rate': 0.01,
    'reg_alpha': 0.3, # L1 Regularization to prevent noise
    'reg_lambda': 0.3, # L2 Regularization
    'importance_type': 'gain',
    'verbosity': -1,
    'random_state': 42
}

model_v4 = lgb.LGBMRegressor(**lgbm_v4_params)

print(f"Training V4 Poisson Ensemble with Season Baseline...")
print(f"Features in use: {list(X_v4.columns)}")

# 4. FIT MODEL
model_v4.fit(
    X_v4, y_v4_clipped, 
    sample_weight=weights,
    categorical_feature=['position', 'next_is_home']
)

# 5. SAVE V4 ARTIFACTS
V4_DIR = Path('../output/forecaster_pro_v4')
V4_DIR.mkdir(parents=True, exist_ok=True)
model_v4.booster_.save_model(str(V4_DIR / 'fpl_pro_model_v4.txt'))

# 6. DIAGNOSTIC PERFORMANCE
y_pred_v4 = model_v4.predict(X_v4)
print(f"\nV4 Performance Audit:")
print(f"MAE (Clipped): {mean_absolute_error(y_v4_clipped, y_pred_v4):.4f}")
print(f"Prediction Mean: {np.mean(y_pred_v4):.3f} (Compared to Actual Mean: {np.mean(y_v4_clipped):.3f})")

# Verify big score accuracy
big_mask = y_v4_clipped >= 10
if big_mask.any():
    print(f"MAE on Big Scores (10+): {mean_absolute_error(y_v4_clipped[big_mask], y_pred_v4[big_mask]):.4f}")

In [ ]:
# %% [Cell 21: GW29 Forecast v4 - FIXED for 12 Features]
import pandas as pd
import numpy as np
from pathlib import Path

# 1. SETUP V4 PATHS
V4_DIR = Path('../output/forecaster_pro_v4')
V4_DIR.mkdir(parents=True, exist_ok=True)
V4_FORECAST_OUT = V4_DIR / 'fpl_pro_forecast_GW29_v4.csv'

# 2. PREPARE INFERENCE DATA (Aligned to 12 features)
inference_v4 = []
names_v4 = []

# Sort by time for recent lags
df_inf = df_augmented.sort_values(['Player UUID', 'season', 'Gameweek'])

for pid, group in df_inf.groupby('Player UUID'):
    if len(group) >= 6:
        # Features: Last 6 points (lags)
        lags = list(group['Total Points'].tail(6).values)
        
        # Context: Get info for NEXT game from the last row
        last_row = group.iloc[-1]
        
        # BUILD THE ROW: Exactly 12 features to match model_v4 training
        row = lags + [
            last_row['Position'],
            last_row['Next_Opponent_Difficulty'],
            last_row['Next_Is_Home'],
            last_row['Rolling_Avg_Minutes_5'],
            last_row['Season_Phase'],
            last_row['Prev_Season_Avg_Points'] # This is the critical 12th feature
        ]
        inference_v4.append(row)
        names_v4.append(last_row['Web Name'])

# 3. PREDICT
# Column names must match the names used in the training window_df
v4_cols = [f'lag_{i}' for i in range(6, 0, -1)] + \
          ['position', 'next_difficulty', 'next_is_home', 'rolling_min', 'season_phase', 'season_baseline']

inf_df_v4 = pd.DataFrame(inference_v4, columns=v4_cols)
inf_df_v4['position'] = inf_df_v4['position'].astype('category')

# Use the 12-feature Poisson model
v4_preds = model_v4.predict(inf_df_v4)

# 4. SAVE LEADERBOARD
leaderboard_v4 = pd.DataFrame({
    'Player': names_v4,
    'Predicted_Points_V4': np.round(v4_preds, 2),
    'Recent_Min_Avg': np.round(inf_df_v4['rolling_min'], 1),
    'Season_Baseline': np.round(inf_df_v4['season_baseline'], 2)
}).sort_values(by='Predicted_Points_V4', ascending=False)

leaderboard_v4.to_csv(V4_FORECAST_OUT, index=False, encoding='utf-8-sig')

print(f"V4 Aggressive Forecast (12 Features) saved to: {V4_FORECAST_OUT}")
display(leaderboard_v4.head(15))

In [ ]:
# %% [Cell 22: Truth Verification v4 - FIXED for 12 Features]
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Setup paths
V4_DIR = Path('../output/forecaster_pro_v4')
V4_DIR.mkdir(parents=True, exist_ok=True)
detailed_v4 = []

# Define the exact 12 columns used in training model_v4
v4_cols_fixed = [f'lag_{i}' for i in range(6, 0, -1)] + \
                ['position', 'next_difficulty', 'next_is_home', 'rolling_min', 'season_phase', 'season_baseline']

print("Analyzing Truth Verification for V4 (Poisson/Baseline Bridge)...")

for pid, group in df_augmented.groupby('Player UUID'):
    group = group.sort_values(['season', 'Gameweek'])
    
    # Needs 7 rows: 6 for history and 1 for the target match
    if len(group) >= 7:
        history = group.iloc[-7:-1]
        target_row = group.iloc[-1]
        
        # BUILD THE VECTOR: Exactly 12 features (Removed V3 noise, added Baseline)
        input_vector = list(history['Total Points'].values) + [
            target_row['Position'], 
            target_row['Next_Opponent_Difficulty'],
            target_row['Next_Is_Home'], 
            target_row['Rolling_Avg_Minutes_5'],
            target_row['Season_Phase'],
            target_row['Prev_Season_Avg_Points'] # The 12th feature
        ]
        
        # Create input DataFrame with aligned columns
        input_df = pd.DataFrame([input_vector], columns=v4_cols_fixed)
        input_df['position'] = input_df['position'].astype('category')
        
        # Prediction
        p_val = model_v4.predict(input_df)[0]
        actual = target_row['Total Points']
        
        detailed_v4.append({
            'Player': target_row['Web Name'],
            'Actual': actual,
            'Predicted': round(float(p_val), 2),
            'Error': round(abs(p_val - actual), 2)
        })

# 2. SAVE AND DISPLAY
truth_v4_df = pd.DataFrame(detailed_v4).sort_values(by='Error', ascending=True)
truth_v4_df.to_csv(V4_DIR / 'fpl_pro_truth_verification_v4.csv', index=False)

mae_v4 = truth_v4_df['Error'].mean()
print(f"V4 Global MAE: {mae_v4:.2f}")

if 'mae_v1' in locals():
    print(f"Comparison -> V1 MAE: {mae_v1:.2f} | V4 MAE: {mae_v4:.2f}")

display(truth_v4_df.head(20))

In [ ]:
# Final Processing: Filtering 0-minute rows and saving the full 2020-2026 dataset

# Define final output path
CLEAN_OUT = OUTPUT_DIR / "dataset_no0min.csv"

print(f"Preparing final dataset from seasons: {df_final['season'].unique()}")

# Filter to keep only rows where players actually played (Minutes > 0)
df_dataset_no0min = df_final[df_final["Minutes Played"] > 0].copy()

print(f"Original records: {len(df_final):,}")
print(f"Remaining records (Minutes > 0): {len(df_dataset_no0min):,}")
print(f"Removed {len(df_final) - len(df_dataset_no0min):,} inactive records.")

# Save to output folder
df_dataset_no0min.to_csv(CLEAN_OUT, index=False, encoding="utf-8-sig")

print(f"SUCCESS!")
print(f"File saved to: {CLEAN_OUT}")
print(f"Final Season Range: {df_dataset_no0min['season'].min()} to {df_dataset_no0min['season'].max()}")
